# Phase 1 Final Feature Manifest

Notebook fix marker: `phase1_final_feature_manifest_v2_common_channels_20260421`

Purpose: record the final scalp feature schema and provenance manifest after the final LOSO split manifest has passed validation. This notebook does **not** write a feature matrix, train models, compute metrics, run leakage audits, or open claims.

Scientific integrity rule: `final_feature_manifest.json` is a schema/provenance artifact only. It cannot be interpreted as decoder evidence or privileged-transfer evidence. The feature schema must use channels common to the final materialized sessions; non-common channels are excluded with provenance instead of being imputed.

## Technical Basis And Boundary

This notebook follows the current repository contract:

- `configs/phase1/final_feature_manifest.json`: feature manifest prerequisites, feature family, windows, bands, trial filter, and non-claim boundary.
- `configs/phase1/final_split_feature_leakage.json`: requires final split, final feature, and final leakage-audit manifests before final comparator outputs are claim-evaluable.
- `notebooks/19_colab_phase1_final_split_manifest.ipynb`: source final LOSO split manifest. It must be ready and validation-passed.
- `src/phase1/final_feature_manifest.py`: fail-closed CLI runner. It writes `final_feature_manifest.json` only when split/Gate0/materialization/dataset sidecar prerequisites pass.

Allowed interpretation: final feature schema/provenance may be used as an input contract for later final feature extraction/comparator runners. It is not feature values, not model output, and not a claim.

In [ ]:
# Cell 1 - Runtime setup, Drive mount, clone/pull repo, and unit tests.
# Tests are run before writing feature manifest artifacts.

from pathlib import Path
import base64
import getpass
import json
import subprocess

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752') if IN_COLAB else Path.cwd()

def run(cmd, cwd=None, check=True):
    display = []
    for item in map(str, cmd):
        display.append('<redacted>' if 'Authorization: Basic' in item else item)
    print('$ ' + ' '.join(display))
    result = subprocess.run(list(map(str, cmd)), cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

if IN_COLAB and not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', 'clone', REPO_URL, REPO_DIR])
elif REPO_DIR.exists():
    print(REPO_DIR)

run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Do not generate or review final feature artifacts until tests pass.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])

In [ ]:
# Cell 2 - Fixed paths and expected locked prereg identity.
# Update FINAL_SPLIT_RUN only when a newer reviewed final split manifest run is intentionally selected.

EXPECTED_PREREG_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
FINAL_SPLIT_RUN = DRIVE_ROOT / 'artifacts/phase1_final_split_manifest/20260421T082928310517Z'
DATASET_ROOT = DRIVE_ROOT / 'data/ds004752'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_feature_manifest'

print('Prereg bundle:', PREREG_BUNDLE)
print('Final split run:', FINAL_SPLIT_RUN)
print('Dataset root:', DATASET_ROOT)
print('Output root:', OUTPUT_ROOT)

In [ ]:
# Cell 3 - Validate prereg and final split source.

import hashlib


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

assert PREREG_BUNDLE.exists(), f'Missing prereg bundle: {PREREG_BUNDLE}'
bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_HASH, 'Prereg identity hash mismatch.'

assert FINAL_SPLIT_RUN.exists(), f'Missing final split run: {FINAL_SPLIT_RUN}'
split_summary = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_summary.json')
split_validation = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_validation.json')
split_manifest = read_json(FINAL_SPLIT_RUN / 'final_split_manifest.json')
split_claim = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_claim_state.json')
print('Final split status:', split_summary.get('status'))
print('Split manifest ready:', split_summary.get('split_manifest_ready'))
print('Split validation:', split_validation.get('status'))
print('Split subjects/folds:', split_summary.get('n_eligible_subjects'), split_summary.get('n_folds'))
assert split_summary.get('status') == 'phase1_final_split_manifest_recorded'
assert split_summary.get('split_manifest_ready') is True
assert split_validation.get('status') == 'phase1_final_split_manifest_validation_passed'
assert split_validation.get('all_eligible_subjects_appear_once_as_outer_test') is True
assert split_validation.get('no_subject_overlap_between_train_and_test') is True
assert split_manifest.get('claim_ready') is False
assert split_manifest.get('smoke_artifacts_promoted') is False
assert split_claim.get('claim_ready') is False
assert DATASET_ROOT.exists(), f'Missing dataset root: {DATASET_ROOT}'

In [ ]:
# Cell 4 - Gate 0/materialization preflight from split manifest provenance.
# The feature manifest runner will independently re-check these conditions.

gate0_run = Path(split_manifest['source_gate0_run'])
print('Gate 0 run from final split manifest:', gate0_run)
assert gate0_run.exists(), f'Missing Gate 0 run: {gate0_run}'
gate0_manifest = read_json(gate0_run / 'manifest.json')
cohort_lock = read_json(gate0_run / 'cohort_lock.json')
materialization_path = gate0_run / 'materialization_report.json'
materialization = read_json(materialization_path) if materialization_path.exists() else {'status': 'missing'}
print('Gate 0 manifest status:', gate0_manifest.get('manifest_status'))
print('Cohort lock status:', cohort_lock.get('cohort_lock_status'))
print('Materialization status:', materialization.get('status'))
print('Gate 0 blockers:', gate0_manifest.get('gate0_blockers', []))

if materialization.get('status') != 'complete':
    print('EXPECTED BLOCKED PATH: materialization is not complete, so final_feature_manifest.json must not be written.')
else:
    print('READY PATH: materialization is complete; CLI may write final_feature_manifest.json after sidecar/event validation.')

In [ ]:
# Cell 5 - Source hash inventory for reproducibility.

HASH_TARGETS = [
    'configs/phase1/final_feature_manifest.json',
    'configs/phase1/final_split_feature_leakage.json',
    'configs/phase1/final_split_manifest.json',
    'src/phase1/final_feature_manifest.py',
    'src/phase1/final_split_manifest.py',
    'src/cli.py',
    'tests/unit/test_phase1_final_feature_manifest.py',
    'notebooks/20_colab_phase1_final_feature_manifest.ipynb',
]
source_hashes = {}
for rel in HASH_TARGETS:
    path = REPO_DIR / rel
    assert path.exists(), f'Missing source target: {rel}'
    source_hashes[rel] = hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps(source_hashes, indent=2))

In [ ]:
# Cell 6 - Run the CLI-backed final feature manifest generator.
# This writes a schema/provenance manifest only; it does not write a feature matrix.

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_feature_manifest',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--final-split-run', str(FINAL_SPLIT_RUN),
    '--dataset-root', str(DATASET_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--feature-config', 'configs/phase1/final_feature_manifest.json',
    '--readiness-config', 'configs/phase1/final_split_feature_leakage.json',
]
run(cmd, cwd=REPO_DIR)
print('Final feature manifest command completed. Review artifacts before downstream use.')

In [ ]:
# Cell 7 - Inspect latest output and verify common artifacts.

latest = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest.exists())
assert latest.exists(), 'No latest.txt written for final feature manifest output.'
run_dir = Path(latest.read_text(encoding='utf-8').strip())
print('Latest final feature output:', run_dir)
assert run_dir.exists(), f'Output run directory does not exist: {run_dir}'

required_common = [
    'phase1_final_feature_manifest_inputs.json',
    'phase1_final_feature_manifest_summary.json',
    'phase1_final_feature_manifest_report.md',
    'phase1_final_feature_manifest_source_links.json',
    'phase1_final_feature_inventory.json',
    'phase1_final_feature_manifest_validation.json',
    'phase1_final_feature_manifest_claim_state.json',
]
for name in required_common:
    path = run_dir / name
    print(name, 'exists =', path.exists())
    assert path.exists(), f'Missing required artifact: {name}'

summary = read_json(run_dir / 'phase1_final_feature_manifest_summary.json')
validation = read_json(run_dir / 'phase1_final_feature_manifest_validation.json')
claim_state = read_json(run_dir / 'phase1_final_feature_manifest_claim_state.json')
inventory = read_json(run_dir / 'phase1_final_feature_inventory.json')
print(json.dumps({
    'run_dir': str(run_dir),
    'summary_status': summary.get('status'),
    'feature_manifest_ready': summary.get('feature_manifest_ready'),
    'claim_ready': summary.get('claim_ready'),
    'headline_phase1_claim_open': summary.get('headline_phase1_claim_open'),
    'feature_set_id': summary.get('feature_set_id'),
    'materialization_status': summary.get('materialization_status'),
    'n_subjects': summary.get('n_subjects'),
    'n_sessions': summary.get('n_sessions'),
    'n_channels': summary.get('n_channels'),
    'n_all_discovered_channels': summary.get('n_all_discovered_channels'),
    'n_excluded_non_common_channels': summary.get('n_excluded_non_common_channels'),
    'excluded_non_common_channels': inventory.get('excluded_non_common_channels'),
    'n_features': summary.get('n_features'),
    'n_event_rows_planned': summary.get('n_event_rows_planned'),
    'feature_manifest_blockers': summary.get('feature_manifest_blockers'),
    'claim_blockers': summary.get('claim_blockers'),
}, indent=2))

assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert summary.get('full_phase1_claim_bearing_run_allowed') is False
assert claim_state.get('claim_ready') is False
assert claim_state.get('headline_phase1_claim_open') is False
assert claim_state.get('smoke_artifacts_promoted') is False

In [ ]:
# Cell 8 - Branch-specific validation: ready manifest or blocked record.

final_manifest_path = run_dir / 'final_feature_manifest.json'
blocked_path = run_dir / 'phase1_final_feature_manifest_blocked.json'

if summary.get('feature_manifest_ready'):
    assert summary.get('status') == 'phase1_final_feature_manifest_recorded'
    assert final_manifest_path.exists(), 'Ready run must write final_feature_manifest.json.'
    assert not blocked_path.exists(), 'Ready run must not write blocked record.'
    feature_manifest = read_json(final_manifest_path)
    assert feature_manifest.get('status') == 'phase1_final_feature_manifest_recorded'
    assert feature_manifest.get('claim_ready') is False
    assert feature_manifest.get('standalone_claim_ready') is False
    assert feature_manifest.get('smoke_feature_rows_allowed_as_final') is False
    assert feature_manifest.get('contains_feature_matrix') is False
    assert feature_manifest.get('contains_model_outputs') is False
    assert feature_manifest.get('contains_metrics') is False
    assert feature_manifest.get('feature_count') == len(feature_manifest.get('feature_names', []))
    assert feature_manifest.get('feature_count', 0) > 0
    assert validation.get('status') == 'phase1_final_feature_manifest_validation_passed'
    assert validation.get('contains_feature_matrix') is False
    assert validation.get('smoke_feature_rows_allowed_as_final') is False
    print('READY REVIEW: final_feature_manifest.json was written as schema/provenance only.')
else:
    assert summary.get('status') == 'phase1_final_feature_manifest_blocked'
    assert not final_manifest_path.exists(), 'Blocked run must not write final_feature_manifest.json.'
    assert blocked_path.exists(), 'Blocked run must write phase1_final_feature_manifest_blocked.json.'
    blocked = read_json(blocked_path)
    assert blocked.get('status') == 'phase1_final_feature_manifest_not_written'
    assert 'final_feature_manifest_missing' in claim_state.get('blockers', [])
    print('BLOCKED REVIEW: final_feature_manifest.json was not written because prerequisites are not met.')

In [ ]:
# Cell 9 - Write a non-claim review note for this run.

from datetime import datetime, timezone

review_status = 'phase1_final_feature_manifest_review_pass_non_claim_ready' if summary.get('feature_manifest_ready') else 'phase1_final_feature_manifest_review_pass_non_claim_blocked'
checks = [
    'required_common_artifacts_present',
    'claim_ready_false',
    'headline_phase1_claim_open_false',
    'smoke_artifacts_not_promoted',
    'feature_inventory_written',
]
if summary.get('feature_manifest_ready'):
    checks.extend([
        'final_feature_manifest_written_as_schema_provenance_only',
        'contains_feature_matrix_false',
        'contains_model_outputs_false',
        'contains_metrics_false',
        'feature_count_matches_feature_names',
    ])
else:
    checks.extend([
        'final_feature_manifest_not_written',
        'blocked_record_written',
    ])

review_note = {
    'status': review_status,
    'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
    'reviewed_run': str(run_dir),
    'scope': 'Phase 1 final feature schema/provenance manifest only',
    'checks_passed': checks,
    'feature_manifest_ready': summary.get('feature_manifest_ready'),
    'feature_set_id': summary.get('feature_set_id'),
    'n_subjects': summary.get('n_subjects'),
    'n_sessions': summary.get('n_sessions'),
    'n_channels': summary.get('n_channels'),
    'n_all_discovered_channels': summary.get('n_all_discovered_channels'),
    'n_excluded_non_common_channels': summary.get('n_excluded_non_common_channels'),
    'excluded_non_common_channels': inventory.get('excluded_non_common_channels'),
    'n_features': summary.get('n_features'),
    'n_event_rows_planned': summary.get('n_event_rows_planned'),
    'feature_manifest_blockers': summary.get('feature_manifest_blockers'),
    'claim_blockers': summary.get('claim_blockers'),
    'allowed_interpretation': claim_state.get('allowed_interpretation'),
    'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    'next_allowed_steps': [
        'If feature manifest is blocked, resolve listed provenance/materialization/dataset blockers first.',
        'If feature manifest is ready, proceed to final leakage audit implementation under the same non-claim discipline.',
        'Do not run claim-bearing final comparators until leakage audit, comparator outputs, controls, calibration, influence and reporting are complete.',
    ],
}
review_path = run_dir / 'phase1_final_feature_manifest_review_note.json'
review_path.write_text(json.dumps(review_note, indent=2), encoding='utf-8')
print('Review note written:', review_path)
print(json.dumps(review_note, indent=2))

In [ ]:
# Cell 10 - Closeout.

print('================ Phase 1 Final Feature Manifest Closeout ================')
print('Notebook fix marker: phase1_final_feature_manifest_v1_20260421')
print('Latest final feature output:', run_dir)
print('Feature manifest ready:', summary.get('feature_manifest_ready'))
print('Feature set:', summary.get('feature_set_id'))
print('Materialization status:', summary.get('materialization_status'))
print('Subjects:', summary.get('n_subjects'))
print('Sessions:', summary.get('n_sessions'))
print('Common channels used:', summary.get('n_channels'))
print('All discovered channels:', summary.get('n_all_discovered_channels'))
print('Excluded non-common channels:', summary.get('n_excluded_non_common_channels'))
print('Features:', summary.get('n_features'))
print('Planned event rows:', summary.get('n_event_rows_planned'))
print('Feature blockers:', summary.get('feature_manifest_blockers'))
print('Claim blockers:', summary.get('claim_blockers'))
print('')
if summary.get('feature_manifest_ready'):
    print('CHECK REQUIRED: final_feature_manifest.json exists as schema/provenance only. Review source links and validation before downstream use.')
else:
    print('BLOCKED: final_feature_manifest.json was not written. Resolve listed blockers before retrying.')
print('')
print('NOT OK TO CLAIM: This feature manifest notebook does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')